In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits


from scipy.interpolate import interp1d

from specula.mmlib.analyse_data import plot_output_data
from specula.mmlib.plot_gain_opt import plot_gain_optimization
from specula.mmlib.utils import reshape_on_mask
from specula.mmlib.compute_iffs import compute_and_save_influence_functions, compute_and_save_dcao_matrix
from check_calibration import check_calibration

from specula.lib.make_mask import make_mask

In [ ]:
pupil = fits.getdata('/raid1/mmenessini/calibration/EKARUS/pupilstop/copernico_pupil_120pixels.fits')

plt.figure()
plt.imshow(pupil,cmap='gray',origin='lower')
plt.colorbar()

In [ ]:
mask = fits.getdata('/raid1/mmenessini/calibration/EKARUS/ifunc/mask.fits')
coords = fits.getdata('/raid1/mmenessini/calibration/EKARUS/ifunc/act_coords.fits')
eigvals = fits.getdata('/raid1/mmenessini/calibration/EKARUS/ifunc/eigenvalues.fits')

plt.figure()
plt.imshow(pupil,cmap='gray',origin='lower')
plt.scatter(coords[0],coords[1],c='red',s=1)

L = 1
plt.figure()
plt.loglog(np.arange(len(eigvals[:-L]))+1,eigvals[:-L])
plt.grid(which='both',alpha=0.3)

In [ ]:
kl_basis = fits.getdata('/raid1/mmenessini/calibration/EKARUS/ifunc/dm468_kl_inv.fits')
fname = '/raid1/mmenessini/calibration/EKARUS/pupilstop/copernico_pupil_120pixels.fits'
hdu = fits.open(fname)
pupil_mask = hdu[1].data

max_modes = min(20, kl_basis.shape[0])

# Create a mask array for display
mode_display = np.zeros((max_modes, pupil_mask.shape[0], pupil_mask.shape[1]))

# Place each mode vector into the 2D pupil shape
idx_mask = np.where(pupil_mask)
mode_ids = np.zeros(max_modes,dtype=int)
for i in range(max_modes//2):
    mode_img = np.zeros(pupil_mask.shape)
    mode_ids[i] = i+1
    mode_img[idx_mask] = kl_basis[i]
    mode_display[i] = mode_img
for i in range(max_modes//2,max_modes):
    mode_img = np.zeros(pupil_mask.shape)
    mode_ids[i] = kl_basis.shape[0]-max_modes+i
    mode_img[idx_mask] = kl_basis[mode_ids[i]]
    mode_display[i] = mode_img

# Plot the reshaped modes
n_rows = int(np.round(np.sqrt(max_modes)))
n_cols = int(np.ceil(max_modes / n_rows))
plt.figure(figsize=(18, 12))
for i in range(max_modes):
    plt.subplot(n_rows, n_cols, i+1)
    plt.imshow(np.ma.masked_array(mode_display[i],mask=1-pupil_mask),origin='lower',cmap='RdBu')
    plt.title(f'Mode {mode_ids[i]}')
    plt.axis('off')
plt.tight_layout()

In [ ]:
iffs = fits.getdata('/raid1/mmenessini/calibration/EKARUS/ifunc/dm468_ifunc.fits')
m2c = fits.getdata('/raid1/mmenessini/calibration/EKARUS/m2c/dm468_m2c.fits')

kl_basis = (iffs @ m2c).T

for i in range(max_modes//2):
    mode_img = np.zeros(pupil_mask.shape)
    mode_ids[i] = i+1
    mode_img[idx_mask] = kl_basis[i]
    mode_display[i] = mode_img
for i in range(max_modes//2,max_modes):
    mode_img = np.zeros(pupil_mask.shape)
    mode_ids[i] = kl_basis.shape[0]-max_modes+i
    mode_img[idx_mask] = kl_basis[mode_ids[i]]
    mode_display[i] = mode_img

# Plot the reshaped modes
n_rows = int(np.round(np.sqrt(max_modes)))
n_cols = int(np.ceil(max_modes / n_rows))
plt.figure(figsize=(18, 12))
for i in range(max_modes):
    plt.subplot(n_rows, n_cols, i+1)
    plt.imshow(np.ma.masked_array(mode_display[i],mask=1-pupil_mask),origin='lower',cmap='RdBu')
    plt.title(f'Mode {mode_ids[i]}')
    plt.axis('off')
plt.tight_layout()

In [ ]:
root_dir = '/raid1/mmenessini/results/EKARUS'
calib_dir = '/raid1/mmenessini/calibration/EKARUS'
plot_output_data(root_dir=root_dir,calib_dir=calib_dir)

In [ ]:

# from specula.mmlib.utils import get_pupil_mask
# pupdatapath = '/raid1/mmenessini/calibration/EKARUS/pupils/pyr_pupdata_40x40.fits'
# pyr_mask = get_pupil_mask(filepath=pupdatapath,npix=120,pyr=True)
# # plt.figure()
# # plt.imshow(pyr_mask,origin='lower')
# # plt.colorbar()

# frame = fits.getdata('/raid1/mmenessini/calibration/EKARUS/frames/pyr3.0_40x40_frame_null.fits')[0]    
# plt.figure()
# plt.imshow(frame/np.max(frame)+pyr_mask,origin='lower',cmap='RdBu')
# plt.colorbar()
# thrp = fits.getdata('/raid1/mmenessini/calibration/EKARUS/slopenulls/pyr3.0_40x40_throughput.fits')
# print(thrp)

In [ ]:

modes = np.array([54,120,400])
seeings = np.array([1.8,2.0,2.2,2.4,2.6])

cv = np.zeros([len(seeings),len(modes),435])

for i,seeing in enumerate(seeings):
    for j,N in enumerate(modes):
        cvec = fits.getdata(f'/raid1/mmenessini/calibration/EKARUS/data/s{seeing:1.1f}_{N:1.0f}modes_corrvec.fits')
        cvec = np.maximum(0.0,cvec)
        cv[i,j,:] = cvec
        fits.writeto(f'/raid1/mmenessini/calibration/EKARUS/data/s{seeing:1.1f}_{N:1.0f}modes_corrvec.fits',cvec,overwrite=True)

x = np.arange(435)+1

for j,N in enumerate(modes):
    plt.figure()
    for i,seeing in enumerate(seeings):
        plt.plot(x,1-cv[i,j,:],label=f'Seeing: {seeing:1.1f}"')
    plt.xscale('log')
    plt.legend()
    plt.title(f'{N:1.0f} modes corrected')
    plt.grid()